In [8]:
import requests

def get_lol_match_history(summoner_name, tagline="NA1", platform="na1", api_key="YOUR_API_KEY"):
    """
    Fetch match history for a League of Legends player (Match v5 API)

    Args:
        summoner_name: Player's Riot ID name (without #tag)
        tagline: Riot ID tagline (e.g. "NA1")
        platform: Platform region (na1, euw1, kr, etc.)
        api_key: Riot Games API key

    Returns:
        List of match IDs
    """
    # Map platform to regional cluster for match v5
    regional_map = {
        "na1": "americas", "br1": "americas", "la1": "americas", "la2": "americas",
        "euw1": "europe", "eun1": "europe", "tr1": "europe", "ru": "europe",
        "kr": "asia", "jp1": "asia",
        "oc1": "sea", "ph2": "sea", "sg2": "sea", "th2": "sea", "tw2": "sea", "vn2": "sea"
    }
    region = regional_map.get(platform, "americas")
    headers = {"X-Riot-Token": api_key}

    # Step 1: Get PUUID via Riot Account API
    account_url = f"https://{region}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{summoner_name}/{tagline}"
    account_resp = requests.get(account_url, headers=headers)
    account_resp.raise_for_status()
    puuid = account_resp.json()["puuid"]

    # Step 2: Get match history using PUUID (Match v5)
    matches_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    matches_params = {"start": 0, "count": 50}
    matches_resp = requests.get(matches_url, headers=headers, params=matches_params)
    matches_resp.raise_for_status()

    return matches_resp.json()


# Usage — note: pass tagline separately, and use header-based auth
match_history = get_lol_match_history(
    summoner_name="sacredswords15",
    tagline="NA1",
    platform="na1",
    api_key="RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40".strip()  # Regenerate at developer.riotgames.com
)

print(match_history)

['NA1_5505092770', 'NA1_5505066777', 'NA1_5505047083', 'NA1_5505026084', 'NA1_5505007074', 'NA1_5504989622', 'NA1_5503949838', 'NA1_5502022714', 'NA1_5500410199', 'NA1_5500390042', 'NA1_5500362923', 'NA1_5499946088', 'NA1_5499926127', 'NA1_5499909510', 'NA1_5499891527', 'NA1_5499594693', 'NA1_5499299146', 'NA1_5499272747', 'NA1_5499240552', 'NA1_5499219783', 'NA1_5498904360', 'NA1_5498873097', 'NA1_5498844454', 'NA1_5498795087', 'NA1_5498747088', 'NA1_5498703911', 'NA1_5498650002', 'NA1_5497374011', 'NA1_5497354466', 'NA1_5497336108', 'NA1_5496580631', 'NA1_5496564432', 'NA1_5496545993', 'NA1_5496542872', 'NA1_5496274223', 'NA1_5496127761', 'NA1_5495787736', 'NA1_5495773022', 'NA1_5495759135', 'NA1_5495749767', 'NA1_5494992804', 'NA1_5494986216', 'NA1_5494827406', 'NA1_5494791740', 'NA1_5494498941', 'NA1_5494481895', 'NA1_5494457075', 'NA1_5494436980', 'NA1_5494413961', 'NA1_5494369024']


In [ ]:
import requests

api_key = "RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40"
headers = {"X-Riot-Token": api_key}
url = "https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/sacredswords15/NA1"

r = requests.get(url, headers=headers)
print(r.status_code)
print(r.json())

200
{'puuid': 'QUHKsuLP3VqPLczikq2CI3rYU3s0FlBkmCEDzkPjN46St8z1m0Cm0g40ehp6WQjF12MxgrLY1x9Lrw', 'gameName': 'sacredswords15', 'tagLine': 'NA1'}


In [7]:
import requests

def get_puuid(summoner_name, tagline="NA1", region="americas", api_key="YOUR_KEY"):
    """
    Get PUUID for a player via Riot Account API
    
    Args:
        summoner_name: Riot ID name (without #tag)
        tagline: Riot ID tagline e.g. "NA1"
        region: Regional cluster (americas, europe, asia, sea)
        api_key: Riot Games API key
    
    Returns:
        puuid string
    """
    headers = {"X-Riot-Token": api_key}
    url = f"https://{region}.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{summoner_name}/{tagline}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    return r.json()["puuid"]


# Usage
puuid = get_puuid("sacredswords15", tagline="NA1", api_key="RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40")
print(puuid)

QUHKsuLP3VqPLczikq2CI3rYU3s0FlBkmCEDzkPjN46St8z1m0Cm0g40ehp6WQjF12MxgrLY1x9Lrw


In [9]:
def is_ranked_solo_duo(match_id, region="americas"):
    """
    Check if a match is a ranked solo/duo game
    
    Args:
        match_id: The match ID to check
        region: Regional cluster (default: "americas")
    
    Returns:
        Boolean indicating if the match is ranked solo/duo
    """
    match_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}"
    match_resp = requests.get(match_url, headers=headers)
    match_resp.raise_for_status()
    
    match_data = match_resp.json()
    queue_id = match_data["info"]["queueId"]
    
    # Queue ID 420 = Ranked Solo/Duo
    return queue_id == 420

# Test with the first match
first_match = match_history[0]
result = is_ranked_solo_duo(first_match)
print(f"{first_match} is ranked solo/duo: {result}")

NA1_5505092770 is ranked solo/duo: True


In [9]:
def get_current_rank(puuid, platform="na1", api_key="YOUR_KEY"):
    headers = {"X-Riot-Token": api_key}
    url = f"https://{platform}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    for entry in r.json():
        if entry["queueType"] == "RANKED_SOLO_5x5":
            return f"{entry['tier']} {entry['rank']} - {entry['leaguePoints']} LP"
    
    return "Unranked"


# Now just two steps instead of three
api_key = "RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40"

puuid = get_puuid("sacredswords15", tagline="NA1", api_key=api_key)
rank = get_current_rank(puuid, platform="na1", api_key=api_key)

print(rank)

DIAMOND II - 64 LP


In [5]:
def convert_rank_to_numeric(rank_str):
    rank, tier, dummy1, lp, dummy2 = rank_str.split()

    print(f"Parsed rank: {rank}, tier: {tier}, LP: {lp}")

    tier_value = {
        "IRON": 0,
        "BRONZE": 400,
        "SILVER": 800,
        "GOLD": 1200,
        "PLATINUM": 1600,
        "DIAMOND": 2000,
        "MASTER": 2400,
        "GRANDMASTER": 2400,
        "CHALLENGER": 2400
    }.get(rank, -1)

    division_value = {
        "IV": 0,
        "III": 100,
        "II": 200,
        "I": 300
    }.get(tier, -1)
    
    lp_value = int(lp)

    if tier_value == -1 or division_value == -1:
        raise ValueError(f"Invalid rank string: {rank_str}")
    return tier_value + division_value + lp_value

# Example usage
rank_str = "GOLD II - 50 LP"
numeric_rank = convert_rank_to_numeric(rank_str)
print(f"{rank_str} converts to numeric value: {numeric_rank}")

Parsed rank: GOLD, tier: II, LP: 50
GOLD II - 50 LP converts to numeric value: 1450


In [10]:
# test on austin

api_key = "RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40"

puuid = get_puuid("sacredswords15", tagline="NA1", api_key=api_key)
rank = get_current_rank(puuid, platform="na1", api_key=api_key)
numeric_rank = convert_rank_to_numeric(rank)

print(numeric_rank)

Parsed rank: DIAMOND, tier: II, LP: 64
2264


In [11]:
def get_champion_mastery_total(puuid, platform="na1", api_key="YOUR_KEY"):
    """Get total champion mastery score across all champions"""
    headers = {"X-Riot-Token": api_key}
    url = f"https://{platform}.api.riotgames.com/lol/champion-mastery/v4/scores/by-puuid/{puuid}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    return r.json()  # returns single integer e.g. 342


def get_summoner_level(puuid, platform="na1", api_key="YOUR_KEY"):
    """Get summoner level"""
    headers = {"X-Riot-Token": api_key}
    url = f"https://{platform}.api.riotgames.com/lol/summoner/v4/summoners/by-puuid/{puuid}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    return r.json()["summonerLevel"]


def get_total_ranked_games(puuid, platform="na1", api_key="YOUR_KEY"):
    """Get total ranked solo/duo games played (wins + losses)"""
    headers = {"X-Riot-Token": api_key}
    url = f"https://{platform}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    for entry in r.json():
        if entry["queueType"] == "RANKED_SOLO_5x5":
            return entry["wins"] + entry["losses"]
    
    return 0


# Usage
api_key = "RGAPI-7a6df684-4c59-44e4-bfdb-0825ce91ef40"
puuid = get_puuid("sacredswords15", tagline="NA1", api_key=api_key)

mastery  = get_champion_mastery_total(puuid, api_key=api_key)
level    = get_summoner_level(puuid, api_key=api_key)
games    = get_total_ranked_games(puuid, api_key=api_key)

print(f"Mastery Score : {mastery}")
print(f"Summoner Level: {level}")
print(f"Ranked Games  : {games}")

Mastery Score : 761
Summoner Level: 580
Ranked Games  : 113


In [12]:
def get_ranked_stats(puuid, platform="na1", api_key="YOUR_KEY"):
    """Get ranked wins, losses, and win rate for solo/duo queue"""
    headers = {"X-Riot-Token": api_key}
    url = f"https://{platform}.api.riotgames.com/lol/league/v4/entries/by-puuid/{puuid}"
    
    r = requests.get(url, headers=headers)
    r.raise_for_status()
    
    for entry in r.json():
        if entry["queueType"] == "RANKED_SOLO_5x5":
            wins   = entry["wins"]
            losses = entry["losses"]
            total  = wins + losses
            winrate = round((wins / total) * 100, 1) if total > 0 else 0
            
            return {
                "wins"   : wins,
                "losses" : losses,
                "total"  : total,
                "winrate": f"{winrate}%"
            }
    
    return {"wins": 0, "losses": 0, "total": 0, "winrate": "0%"}


# Usage
stats = get_ranked_stats(puuid, api_key=api_key)
print(f"Wins   : {stats['wins']}")
print(f"Losses : {stats['losses']}")
print(f"Total  : {stats['total']}")
print(f"Winrate: {stats['winrate']}")

Wins   : 53
Losses : 60
Total  : 113
Winrate: 46.9%
